In [1]:
import pandas as pd
import numpy as np

In [2]:
renewal_calls = pd.read_csv('../../data/raw/renewal_calls.csv', low_memory=False)
renewal_calls.columns = (renewal_calls.columns.str.strip().str.lower().str.replace(' ', '_'))
print(f'Columns after renaming:\n {renewal_calls.columns}')

Columns after renaming:
 Index(['call_id', 'call_direction', 'co_ref', 'call_date', 'churn_category',
       'complaint_category', 'customer_reaction_category',
       'agent_renewal_pitch_category', 'customer_renewal_response_category',
       'agent_response_category', 'membership_renewal_decision',
       'serious_complaint', 'other_complaint', 'discussion_on_price_increase',
       'renewal_impact_due_to_price_increase', 'discount_or_waiver_requested',
       'call_reschedule_request', 'agent_flagged_membership_status_alert',
       'agent_renewal_initiation', 'explicit_competitor_mention',
       'unnamed:_20', 'explicit_switching_intent', 'mentioned_competitors',
       'price_switching_mentioned', 'competitor_value_comparison',
       'competitor_benefits_mentioned', 'topic_introduced_by',
       'percentage_price_increase_mentioned',
       'monetary_price_increase_mentioned', 'price_range_mentioned',
       'customer_asked_for_justification', 'customer_response',
       'desir

Dropping Full Empty Columns

In [3]:
empty_cols = renewal_calls.columns[renewal_calls.isnull().all()].tolist()
print(f'Fully empty columns to drop: {empty_cols}')

renewal_calls = renewal_calls.drop(columns=empty_cols)
print(f'Shape after dropping empty columns: {renewal_calls.shape}')

Fully empty columns to drop: ['unnamed:_20']
Shape after dropping empty columns: (186534, 40)


Removing duplicates

In [4]:
print(f'Duplicate rows before: {renewal_calls.duplicated().sum():,}')
renewal_calls = renewal_calls.drop_duplicates()
print(f'Shape after dedup: {renewal_calls.shape}')

Duplicate rows before: 28,812


Shape after dedup: (157722, 40)


Dropping rows where co_ref is null because it acts as primary key to join with other datasets

In [5]:
renewal_calls['co_ref'].isnull().sum()

np.int64(4453)

In [6]:
renewal_calls = renewal_calls.dropna(subset=['co_ref'])
renewal_calls.shape

(153269, 40)

Standardizing call_direction column values

In [7]:
renewal_calls['call_direction'].value_counts()

call_direction
OUT_BOUND    65852
Outbound     65616
IN_BOUND     11149
Inbound      10652
Name: count, dtype: int64

In [8]:
renewal_calls['call_direction'] = renewal_calls['call_direction'].replace({
    'OUT_BOUND': 'Outbound',
    'IN_BOUND': 'Inbound'
})
renewal_calls['call_direction'].value_counts()

call_direction
Outbound    131468
Inbound      21801
Name: count, dtype: int64

Converting proper datatype of dates

In [9]:
renewal_calls['call_date'].dtype

<StringDtype(storage='python', na_value=nan)>

In [10]:
renewal_calls['call_date'] = pd.to_datetime(renewal_calls['call_date'], format='%d-%m-%Y')
renewal_calls['call_date'].dtype

dtype('<M8[us]')

In [11]:
renewal_calls['call_date'].head(5)

0   2025-01-29
1   2025-02-26
2   2025-01-24
3   2025-06-09
4   2024-08-20
Name: call_date, dtype: datetime64[us]

Not all calls were analysed. The `analysed_call` column flags this:<br>
- `1` -> call was fully transcribed and analysed -> all detail columns are populated<br>
- `NaN` -> only the basic call log was recorded -> all detail columns are NaN.<br>

In [12]:
# create a binary flag: 1 = analysed, 0 = call-log-only
renewal_calls['is_analysed'] = (renewal_calls['analysed_call'] == 1).astype(int)

analysed_count     = (renewal_calls['is_analysed'] == 1).sum()
not_analysed_count = (renewal_calls['is_analysed'] == 0).sum()

print(f'Analysed rows: {analysed_count:,}')
print(f'Non-analysed rows: {not_analysed_count:,}')
print()
print('Columns available in non-analysed rows:')
not_analysed = renewal_calls[renewal_calls['is_analysed'] == 0]
available_in_non_analysed = not_analysed.columns[not_analysed.notnull().any()].tolist()
print(available_in_non_analysed)

Analysed rows: 84,675
Non-analysed rows: 68,594

Columns available in non-analysed rows:
['call_id', 'call_direction', 'co_ref', 'call_date', 'call_number', 'call_year', 'is_analysed']


In [13]:
# measure missingness among analysed rows only
analysed_only = renewal_calls[renewal_calls['is_analysed'] == 1]
miss_in_analysed = (analysed_only.isnull().mean() * 100).sort_values(ascending=False)

print('Missingness % among analysed rows (top 15):')
print(miss_in_analysed.head(15).round(1).to_string())

Missingness % among analysed rows (top 15):
argument_that_convinced_customer_to_stay_category    97.9
agent_response_to_cancel_category                    94.3
churn_category                                       90.9
justification_category                               85.1
reason_for_renewal_category                          81.7
complaint_category                                   78.8
customer_reaction_category                           73.9
agent_renewal_pitch_category                         38.6
agent_response_category                              38.4
customer_renewal_response_category                   38.0
other_complaint                                       5.0
serious_complaint                                     5.0
competitor_benefits_mentioned                         2.5
membership_renewal_decision                           2.4
renewal_impact_due_to_price_increase                  0.8


In [14]:
# dropping Columns that have more than 80% empty data and are not analysed
threshold = 80
drop_80 = miss_in_analysed[miss_in_analysed > threshold].index.tolist()

# dropping call_year as it is redundant
# dropping analysed_call as it is replaced by flag
drops = ['call_year', 'analysed_call']
drops.extend(drop_80)
print(drop_80)

renewal_calls = renewal_calls.drop(columns=drops)
print(f'\nShape after dropping: {renewal_calls.shape}')

['argument_that_convinced_customer_to_stay_category', 'agent_response_to_cancel_category', 'churn_category', 'justification_category', 'reason_for_renewal_category']

Shape after dropping: (153269, 34)


## Clean Signal Columns — Analysed Rows Only

For `Yes/No columns`: check if the value **starts with 'Yes'** and ignore the long reasoning string, else map to 0.

For `desire_to_cancel`: 294 unique values. Map to three categories.

Dropping irrelevant columns<br>
call_year -> can be derived from call_date

In [15]:
def extract_yes_no(series, col_name):
    cleaned = series.astype(str).str.strip().str.lower()
    result  = cleaned.str.startswith('yes').astype('Int64')  # Int64 preserves NaN
    # Rows that were 'nan' (NaN cast to string) should stay NaN, not become 0
    result[series.isna()] = pd.NA
    return result


def map_desire_to_cancel(series):
    """
      'cancel'       —  'Desired to Cancel'
      'renew'        —   'Renewed' / 'Renew' (no cancel mention)
      'not_discussed' —  Not Discussed, Not Applicable, NaN
    """
    series_s = series.astype(str).str.lower()
    cancel = series_s.str.contains('desired to cancel', na=False)
    renew  = series_s.str.contains(r'\brenew', na=False) & ~cancel

    out = pd.Series('not_discussed', index=series.index, dtype='object')
    out[renew]  = 'renew'
    out[cancel] = 'cancel'
    out[series.isna()] = np.nan  # keep original NaN as NaN
    return out


In [16]:
renewal_calls['desire_to_cancel_clean'] = map_desire_to_cancel(
    renewal_calls['desire_to_cancel']
)

In [17]:
binary_cols = {
    'serious_complaint':                    'serious_complaint_flag',
    'other_complaint':                      'other_complaint_flag',
    'discussion_on_price_increase':         'price_discussed_flag',
    'renewal_impact_due_to_price_increase': 'price_impact_renewal_flag',
    'discount_or_waiver_requested':         'discount_requested_flag',
    'call_reschedule_request':              'call_reschedule_flag',
    'agent_flagged_membership_status_alert':'membership_alert_flag',
    'agent_renewal_initiation':             'agent_initiated_renewal_flag',
    'explicit_competitor_mention':          'competitor_mentioned_flag',
    'explicit_switching_intent':            'switching_intent_flag',
    'price_switching_mentioned':            'price_switching_flag',
    'percentage_price_increase_mentioned':  'pct_price_increase_flag',
    'monetary_price_increase_mentioned':    'monetary_price_increase_flag',
    'customer_asked_for_justification':     'customer_asked_justification_flag',
    'discount_offered':                     'discount_offered_flag',
    'membership_renewal_decision':          'renewal_confirmed_flag',
}

for a, b in binary_cols.items():
    if a in renewal_calls.columns:
        renewal_calls[b] = extract_yes_no(renewal_calls[a], a)

In [18]:
# Neutral / Positive / Negative / Not Discussed Mapping
if 'customer_response' in renewal_calls.columns:
    response_map = {'Positive': 1, 'Neutral': 0, 'Negative': -1, 'Not Discussed': np.nan}
    renewal_calls['customer_response_score'] = (
        renewal_calls['customer_response']
        .astype(str).str.strip()
        .map(response_map)
    )
    print(renewal_calls['customer_response_score'].value_counts(dropna=False))

customer_response_score
 NaN    90932
 0.0    55968
-1.0     5652
 1.0      717
Name: count, dtype: int64


In [19]:
print(f'Final shape: {renewal_calls.shape}')
print(f'\nAnalysed rows: {(renewal_calls["is_analysed"]==1).sum():,}')
print(f'Non-analysed rows: {(renewal_calls["is_analysed"]==0).sum():,}')
print(f'\nAll columns:')
for i, col in enumerate(renewal_calls.columns, 1):
    dtype = renewal_calls[col].dtype
    nulls = renewal_calls[col].isnull().sum()
    print(f'  {i:2d}. {col} ({dtype}) — {nulls:,} nulls')

Final shape: (153269, 52)

Analysed rows: 84,675
Non-analysed rows: 68,594

All columns:
   1. call_id (float64) — 0 nulls
   2. call_direction (str) — 0 nulls
   3. co_ref (str) — 0 nulls
   4. call_date (datetime64[us]) — 0 nulls
   5. complaint_category (str) — 135,320 nulls
   6. customer_reaction_category (str) — 131,194 nulls
   7. agent_renewal_pitch_category (str) — 101,259 nulls
   8. customer_renewal_response_category (str) — 100,765 nulls
   9. agent_response_category (str) — 101,088 nulls
  10. membership_renewal_decision (str) — 70,634 nulls
  11. serious_complaint (str) — 72,837 nulls
  12. other_complaint (str) — 72,838 nulls
  13. discussion_on_price_increase (str) — 69,252 nulls
  14. renewal_impact_due_to_price_increase (str) — 69,278 nulls
  15. discount_or_waiver_requested (str) — 69,278 nulls
  16. call_reschedule_request (str) — 68,751 nulls
  17. agent_flagged_membership_status_alert (str) — 68,751 nulls
  18. agent_renewal_initiation (str) — 68,751 nulls
  19. e

Saving cleaned dataset

In [20]:
import os
os.makedirs('../../data/cleaned', exist_ok=True)
renewal_calls.to_csv('../../data/cleaned/renewal_calls_cleaned.csv', index=False)
print('Saved! Shape:', renewal_calls.shape)

Saved! Shape: (153269, 52)
